In [51]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
from src.model import GPT
import tiktoken 
import torch
import torch.nn as nn
from torch.nn import functional as F 

In [53]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'mps'

In [54]:
num_return_sequences= 5
max_length = 30

### Loading The Model

In [55]:
model = GPT.from_pretrained('gpt2')
print(f"Model Loaded: {model}")
model.eval()
model = model.to(device)

loading wights from pretrained gpt: gpt2
Model Loaded: GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


#### Doing a forward Pass

In [77]:
x = "Hi, My name is Vedaang Chopra."
tokenizer = tiktoken.get_encoding("gpt2")
enc_x = tokenizer.encode(x)

print(enc_x, type(enc_x))

[17250, 11, 2011, 1438, 318, 569, 18082, 648, 40159, 430, 13] <class 'list'>


In [78]:
enc_x = torch.tensor(enc_x, dtype = torch.long)
enc_x = enc_x.unsqueeze(0).repeat(num_return_sequences, 1)
enc_x= enc_x.to(device)

#### Implementing the Prefill and Decode

In [79]:
print(enc_x.shape)

torch.Size([5, 11])


In [80]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
print(x)

Hi, My name is Vedaang Chopra.


In [ ]:
while enc_x.size(1) < max_length:
    with torch.no_grad():
        loss, logits = model(enc_x)
        # print(logits.shape)
        
        logits = logits[: , -1, :]
        
        probs = F.softmax(logits, dim=-1)
        # print(probs.shape)
        
        topk_probs , topk_indices = torch.topk(probs, 50, dim=-1)
        
        ix = torch.multinomial(topk_probs, 1)
        
        # print(ix.shape, ix.squeeze(1).shape) 
        
        
        xcol = torch.gather(topk_indices, -1, ix)
        
        # print(xcol.shape)
        # print(tokenizer.decode(xcol.squeeze(1).tolist()), end='')
        
        enc_x = torch.cat((enc_x, xcol), dim=1)
        # break

In [82]:
for i in range(num_return_sequences):
    tokens = enc_x[i, :max_length].tolist()
    decoded = tokenizer.decode(tokens)
    print(">", decoded)

> Hi, My name is Vedaang Chopra. And why he never bothered with its a-2 = 2. 2: $g g —
> Hi, My name is Vedaang Chopra. I was able to speak with you.











> Hi, My name is Vedaang Chopra. I'm gonna go through a lot the one-on the P. The Lang Chi at
> Hi, My name is Vedaang Chopra. I think she-Waters, 13 : 10 : (E : AS POS N? AS
> Hi, My name is Vedaang Chopra.d - There is good forked out toto be on the new season is the first .
